<h1>Introduction Summary</h1>
You are building an NLP chatbot capable of slot-filling for natural language understanding (NLU) tasks. The chatbot uses a deep learning model based on BERT for token classification. A BIO-tagged dataset is used to train the model to identify and extract information (slots) from user input. Key steps include preprocessing data, tokenizing inputs, aligning labels, creating datasets, and fine-tuning a BERT model for sequence tagging. The trained model can predict slot information in new sentences for chatbot functionalities.

<h1>1. Ensure the Dataset is Clean</h1>
Ensure your dataset (crafted_dataset_1k.csv) has:

* sentence: Input text (full sentences).
* bio_tags: Corresponding BIO tags aligned with tokens.
* tokens: Tokens for each sentence, comma-separated.
Preview a few rows to confirm the structure:

In [1]:
import pandas as pd

# Load dataset
file_path = "crafted_dataset_10k.csv"
data = pd.read_csv(file_path)

# Preview data
print(data.head())


                                            sentence  \
0              need {route_length: 800 meters} track   
1  {start_location: near me} to {end_location: me...   
2  I'm staying at {start_location: shimshon} {loc...   
3  Plan a route from {start_location: king george...   
4  run from {start_location: acre} to {end_locati...   

                                              tokens  \
0                           need, 800, meters, track   
1  near, me, to, meah, shearim, street, 40, ,, no...   
2  I'm, staying, at, shimshon, 11, and, would, lo...   
3  Plan, a, route, from, king, george, street, 9,...   
4                   run, from, acre, to, hapisga, 16   

                                            bio_tags  
0               O, B-route_length, I-route_length, O  
1  B-start_location, I-start_location, O, B-end_l...  
2  O, O, O, B-start_location, B-loca_start_num, O...  
3  O, O, O, O, B-start_location, I-start_location...  
4  O, O, B-start_location, O, B-end_location, B-l..

<h1>2. Split the Dataset</h1>
Split the dataset into training, validation, and testing sets:

In [2]:
from sklearn.model_selection import train_test_split

# Split into train, validation, and test sets
train_data, temp_data = train_test_split(data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)


<h1>3. Preprocess Data</h1>
Extract tokens and tags, converting them into lists of lists:

In [3]:
def preprocess_data(data):
    sentences = data["tokens"].apply(lambda x: x.split(',')).tolist()
    tags = data["bio_tags"].apply(lambda x: x.split(',')).tolist()
    return sentences, tags

train_sentences, train_tags = preprocess_data(train_data)
val_sentences, val_tags = preprocess_data(val_data)
test_sentences, test_tags = preprocess_data(test_data)


<h1>4. Define Tag Mapping</h1>
Create mappings for BIO tags:

In [4]:
tag2id = {
    "O": 0, "B-difficulty": 1, "I-difficulty": 2,
    "B-route_length": 3, "I-route_length": 4,
    "B-start_location": 5, "I-start_location": 6,
    "B-start_number": 7, "I-start_number": 8,
    "B-end_location": 9, "I-end_location": 10,
    "B-loca_end_num": 11, "I-loca_end_num": 12,
    "B-loca_start_num": 13, "I-loca_start_num": 14,
}
id2tag = {v: k for k, v in tag2id.items()}

<h1>5. Tokenize and Align Labels</h1>
Ensure tokens and labels are properly aligned using word_ids from the tokenizer:

In [5]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(sentences, tags):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )

    labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)  # Ignore special tokens
            elif word_id < len(label):
                aligned_labels.append(tag2id[label[word_id].strip()])
            else:
                aligned_labels.append(-100)  # In case of mismatch
        labels.append(aligned_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize datasets
train_inputs = tokenize_and_align_labels(train_sentences, train_tags)
val_inputs = tokenize_and_align_labels(val_sentences, val_tags)
test_inputs = tokenize_and_align_labels(test_sentences, test_tags)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

<h1>6. Create PyTorch Dataset</h1>
Define a dataset class:

In [6]:
import torch
from torch.utils.data import Dataset

class TokenClassificationDataset(Dataset):
    def __init__(self, inputs):
        self.inputs = inputs

    def __len__(self):
        return len(self.inputs["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.inputs["input_ids"][idx],
            "attention_mask": self.inputs["attention_mask"][idx],
            "labels": self.inputs["labels"][idx],
        }

train_dataset = TokenClassificationDataset(train_inputs)
val_dataset = TokenClassificationDataset(val_inputs)
test_dataset = TokenClassificationDataset(test_inputs)


<h1>7. Initialize and Train the Model</h1>
Load the pre-trained BERT model and set training arguments:

In [7]:
# API_KEY: 611797cd91efcf043036bb77a209ff83138c41e3
from transformers import BertForTokenClassification, TrainingArguments, Trainer

model = BertForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(tag2id))

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    save_steps=500,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

trainer.train()


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-7-c236f70e2027>:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch,Training Loss,Validation Loss
1,No log,0.018986
2,0.164300,0.003617
3,0.006100,0.002180


TrainOutput(global_step=1314, training_loss=0.06517800618433699, metrics={'train_runtime': 299.7214, 'train_samples_per_second': 70.065, 'train_steps_per_second': 4.384, 'total_flos': 610955036370000.0, 'train_loss': 0.06517800618433699, 'epoch': 3.0})

<h1>8. Evaluate the Model</h1>
Use the test dataset and the seqeval library for evaluation:

In [8]:
!pip install seqeval
from seqeval.metrics import classification_report

# Get predictions
predictions, labels, _ = trainer.predict(test_dataset)

# Convert predictions and true labels back to tags
predicted_tags = [[id2tag[id] for id in pred if id != -100] for pred in predictions.argmax(axis=2)]
true_tags = [[id2tag[id] for id in label if id != -100] for label in labels]

true_lengths = [len(seq) for seq in true_tags]
pred_lengths = [len(seq) for seq in predicted_tags]

# Truncate predictions to match the true tag lengths
adjusted_predicted_tags = [
    pred[:len(true)]
    for pred, true in zip(predicted_tags, true_tags)
]

# Verify the lengths are now aligned
true_lengths = [len(seq) for seq in true_tags]
adjusted_pred_lengths = [len(seq) for seq in adjusted_predicted_tags]

print(f"True lengths: {true_lengths}")
print(f"Adjusted predicted lengths: {adjusted_pred_lengths}")



# Evaluate using seqeval
# print(classification_report(true_tags, predicted_tags))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=4d971d00fdd056f13675df17bdc6d716d69ea8b7c279e9d6bbc4982aa1657261
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval


True lengths: [4, 18, 19, 13, 5, 16, 8, 7, 10, 4, 12, 7, 13, 12, 8, 10, 26, 5, 16, 5, 8, 11, 16, 14, 20, 8, 15, 15, 28, 6, 8, 5, 5, 19, 7, 5, 17, 10, 13, 26, 5, 19, 10, 10, 27, 19, 28, 34, 35, 16, 11, 5, 23, 15, 22, 18, 19, 17, 34, 4, 22, 21, 18, 17, 5, 14, 9, 21, 15, 8, 25, 8, 23, 8, 28, 6, 12, 27, 19, 20, 23, 23, 7, 14, 6, 6, 7, 6, 8, 17, 14, 13, 7, 9, 11, 11, 17, 14, 18, 6, 27, 6, 9, 10, 22, 6, 34, 9, 20, 5, 8, 8, 21, 6, 10, 12, 19, 20, 5, 23, 6, 10, 15, 23, 8, 9, 17, 6, 16, 8, 5, 5, 7, 6, 13, 11, 12, 13, 12, 16, 17, 21, 15, 18, 17, 8, 22, 23, 18, 20, 7, 12, 11, 9, 20, 14, 18, 20, 34, 19, 14, 9, 12, 4, 17, 10, 18, 6, 10, 4, 25, 5, 20, 27, 19, 8, 11, 5, 24, 10, 18, 8, 17, 21, 13, 25, 6, 30, 26, 17, 10, 24, 18, 8, 18, 7, 4, 8, 23, 20, 19, 28, 12, 5, 28, 4, 24, 9, 10, 12, 15, 4, 7, 11, 29, 26, 6, 14, 7, 27, 22, 20, 12, 15, 7, 12, 10, 12, 17, 9, 16, 6, 14, 11, 16, 11, 5, 4, 15, 25, 5, 8, 7, 17, 13, 8, 4, 6, 10, 15, 9, 10, 23, 20, 6, 9, 27, 19, 5, 15, 19, 11, 22, 9, 7, 6, 19, 21, 19, 6, 

<h1>9. Inference</h1>
To use the model for slot filling on new sentences:

In [9]:
import torch

def predict_slots(sentence):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Check the available device
    model.to(device)  # Move the model to the device

    tokens = sentence.split()  # Tokenize sentence
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}  # Move inputs to the same device as the model

    with torch.no_grad():  # Ensure no gradients are computed
        outputs = model(**inputs)

    predictions = outputs.logits.argmax(dim=2).squeeze().tolist()
    predicted_tags = [id2tag[p] for p in predictions if p != -100]
    return list(zip(tokens, predicted_tags))


print(predict_slots("I am looking to run from haneviim 24, 10 km long and end at herzel 45. and i want a hard route"))


[('I', 'O'), ('am', 'O'), ('looking', 'O'), ('to', 'O'), ('run', 'O'), ('from', 'O'), ('haneviim', 'O'), ('24,', 'B-start_location'), ('10', 'B-start_location'), ('km', 'B-start_location'), ('long', 'B-start_location'), ('and', 'B-loca_start_num'), ('end', 'B-route_length'), ('at', 'B-route_length'), ('herzel', 'I-route_length'), ('45.', 'I-route_length'), ('and', 'O'), ('i', 'O'), ('want', 'O'), ('a', 'B-end_location'), ('hard', 'B-end_location'), ('route', 'B-loca_end_num')]


<h3>This step-by-step process ensures:</h3>
<br>1. Data is clean and correctly tokenized.
<br>2. Tokens and tags are properly aligned.
<br>3. Model training and evaluation are seamless.fine-tune if necessary.